# 02 · Experiments

3 representations × 3 architectures, each evaluated with k-fold cross-validation.
Every run saves its fold scores to `results/<architecture>__<representation>.json`.

In [29]:
# Setup: make the course code (part1) and our code (src) importable.
# Run notebooks from this folder (JupyterLab / VS Code do this by default).
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [30]:
from part1 import experiment
from src import config, cv, data, encoders, results_io

## Data and folds

In [31]:
LABEL_MODE = "fine"   # "fine" (28) | "ekman" (7) | "sentiment" (4)
pairs, outcome_space, stats = data.load_pairs(label_mode=LABEL_MODE)
print(stats)
folds = experiment.cross_validation_split(pairs, k=config.K_FOLDS, seed=config.SEED)

{'label_mode': 'fine', 'min_raters': 2, 'comments_with_label': 54263, 'kept': 45446, 'dropped_multilabel': 8817}


## Tokeniser

Reuse the tutorial's or train our own? On what corpus, which vocab size?
Note the choice + reason in the decisions log.

In [32]:
corpus = data.tokenizer_corpus()
tokenizer = experiment.train_tokenizer(corpus, vocab_size=10_000, min_frequency=2)  # TODO: tune
tokenizer.get_vocab_size()

10000

## Representations

In [33]:
representations = {
    "indicator": encoders.TokenIndicator(tokenizer),
    "bow": encoders.BagOfWords(tokenizer),
    # Third: TF-IDF. IDF weights are fitted per fold on its training texts only (no leakage),
    # so this is a {fold: encoder} dict; the CV functions below accept both forms.
    "tfidf": encoders.fit_per_fold(folds, lambda: encoders.TfIdf(tokenizer)),
}

## Log-linear

In [34]:
for rep_name, enc in representations.items():
    name = f"loglin__{rep_name}"
    out = cv.course_cv(folds, outcome_space, enc, "logistic", save_to=str(config.MODELS_DIR / name))
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size())
    print(name, out["accuracies"])

  0%|          | 0/5 [00:00<?, ?it/s]

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: T

loglin__indicator OrderedDict({0: 0.5297029702970297, 1: 0.5458246231708659, 2: 0.5319617119595115, 3: 0.5384530751457807, 4: 0.5310815271206953})


  0%|          | 0/5 [00:00<?, ?it/s]

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: T

loglin__bow OrderedDict({0: 0.5006600660066006, 1: 0.5318516888546595, 2: 0.505996259214435, 3: 0.5106172296182199, 4: 0.5046759819562108})


... (8 outputs left out)

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  0%|          | 0/1 [00:00<?, ?it/s]

/Users/nordin/UNI-LOCAL/MASTER/JAAR-1/NLP/nlp1-rosa-nordin/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1435: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


loglin__tfidf OrderedDict({0: 0.5138613861386139, 1: 0.5371327978875564, 2: 0.517768731433601, 3: 0.5239300253053141, 4: 0.5152382000220046})


## MLP (one hidden layer)

In [35]:
smlp_args = dict(hidden_dim=128, dropout=0.1, num_epochs=10)  # TODO: tune

for rep_name, enc in representations.items():
    name = f"mlp__{rep_name}"
    out = cv.course_cv(folds, outcome_space, enc, "mlp", save_to=str(config.MODELS_DIR / name),
                       smlp_args=smlp_args)
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size(), **smlp_args)
    print(name, out["accuracies"])

  0%|          | 0/5 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 0.876 seconds
Dataloader done! Time taken: 1.196 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.2315 | train acc: 0.4354 | val loss: 1.8244 | val acc: 0.5107
Epoch 02 | train loss: 1.3894 | train acc: 0.6028 | val loss: 1.6571 | val acc: 0.5404
Epoch 03 | train loss: 0.9599 | train acc: 0.7133 | val loss: 1.6859 | val acc: 0.5424
Epoch 04 | train loss: 0.6889 | train acc: 0.7938 | val loss: 1.8169 | val acc: 0.5329
Epoch 05 | train loss: 0.5071 | train acc: 0.8519 | val loss: 1.9809 | val acc: 0.5201
Epoch 06 | train loss: 0.3785 | train acc: 0.8952 | val loss: 2.1479 | val acc: 0.5101
Epoch 07 | train loss: 0.2861 | train acc: 0.9232 | val loss: 2.3344 | val acc: 0.5036
Epoch 08 | train loss: 0.2226 | train acc: 0.9434 | val loss: 2.5549 | val acc: 0.4987
Epoch 09 | train loss: 0.1758 | train acc: 0.9576 | val loss: 2.7342 | val acc: 0.4975
Epoch 10 | train loss: 0.1393 | train acc: 0.9671 | val loss: 2.9079 | val acc: 0.4861
Training done! Time taken: 16.564 seconds
Transforming documents...
Document transformation done! Time taken: 0.847 seconds
Datalo

  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.2524 | train acc: 0.4311 | val loss: 1.7640 | val acc: 0.5356
Epoch 02 | train loss: 1.4035 | train acc: 0.5955 | val loss: 1.5740 | val acc: 0.5642
Epoch 03 | train loss: 0.9701 | train acc: 0.7070 | val loss: 1.6025 | val acc: 0.5617
Epoch 04 | train loss: 0.6975 | train acc: 0.7891 | val loss: 1.7186 | val acc: 0.5525
Epoch 05 | train loss: 0.5160 | train acc: 0.8485 | val loss: 1.8527 | val acc: 0.5401
Epoch 06 | train loss: 0.3893 | train acc: 0.8899 | val loss: 2.0395 | val acc: 0.5365
Epoch 07 | train loss: 0.2990 | train acc: 0.9190 | val loss: 2.2236 | val acc: 0.5299
Epoch 08 | train loss: 0.2340 | train acc: 0.9377 | val loss: 2.3873 | val acc: 0.5205
Epoch 09 | train loss: 0.1851 | train acc: 0.9545 | val loss: 2.5642 | val acc: 0.5191
Epoch 10 | train loss: 0.1479 | train acc: 0.9644 | val loss: 2.7513 | val acc: 0.5163
Training done! Time taken: 15.438 seconds
Transforming documents...
Document transformation done! Time taken: 0.909 seconds
Datalo

... (35 outputs left out)

  0%|          | 0/1 [00:00<?, ?it/s]

Transforming documents...
Document transformation done! Time taken: 1.830 seconds
Dataloader done! Time taken: 1.204 seconds
Training...


  0%|          | 0/10 [00:00<?, ?it/s]

Epoch 01 | train loss: 2.4410 | train acc: 0.3865 | val loss: 2.0660 | val acc: 0.4549
Epoch 02 | train loss: 1.7302 | train acc: 0.5276 | val loss: 1.7655 | val acc: 0.5206
Epoch 03 | train loss: 1.3193 | train acc: 0.6203 | val loss: 1.6662 | val acc: 0.5392
Epoch 04 | train loss: 1.0353 | train acc: 0.6943 | val loss: 1.6701 | val acc: 0.5348
Epoch 05 | train loss: 0.8260 | train acc: 0.7549 | val loss: 1.7242 | val acc: 0.5299
Epoch 06 | train loss: 0.6660 | train acc: 0.8055 | val loss: 1.8085 | val acc: 0.5225
Epoch 07 | train loss: 0.5358 | train acc: 0.8479 | val loss: 1.9061 | val acc: 0.5136
Epoch 08 | train loss: 0.4357 | train acc: 0.8820 | val loss: 2.0021 | val acc: 0.5009
Epoch 09 | train loss: 0.3527 | train acc: 0.9084 | val loss: 2.1452 | val acc: 0.4964
Epoch 10 | train loss: 0.2875 | train acc: 0.9312 | val loss: 2.2704 | val acc: 0.4970
Training done! Time taken: 15.072 seconds
mlp__tfidf OrderedDict({0: 0.5355335533553355, 1: 0.5553966332929915, 2: 0.5390031906700

## Our third architecture: deep low-rank MLP

`src/models.py::DeepLowRankMLP`: several hidden layers, each weight matrix factorised as W = A·B with rank r
(the LoRA factorisation, trained from scratch: there is no pretrained model to adapt).
The first layer (vocab × hidden) holds most parameters; the rank bottleneck shrinks it so we can stack more layers.
Residual connections between equal-width layers keep the deep network trainable.

In [39]:
import numpy as np
import pandas as pd
from src import models

# Quick check before the full CV run: parameter counts of full-rank vs low-rank
V, C = tokenizer.get_vocab_size(), len(outcome_space)
for r in [None, 64, 16]:
    m = models.DeepLowRankMLP(V, [256, 256, 256, 256], C, rank=r)
    print(f"rank={r}: {models.count_parameters(m):,} parameters")

rank=None: 2,764,828 parameters
rank=64: 762,908 parameters
rank=16: 196,892 parameters


In [37]:
deep_args = dict(hidden_dims=[256, 256, 256, 256], rank=32, dropout=0.5, residual=True)  # TODO: tune
# No early stopping: every fold trains all epochs, so the table shows where it starts to overfit.
# The reported CV accuracy is still that of the checkpoint with the lowest val loss (as in part1.smlp).
deep_train_args = dict(num_epochs=10, learning_rate=1e-3, batch_size=128, seed=config.SEED,
                       device="auto")   # "auto" = CUDA if available, else CPU (also on a Mac)
N_JOBS = None   # parallel training processes; None = half the cores on CPU, 1 on a GPU; 1 = sequential

outs = models.run_experiments(
    {f"deeplowrank__{rep}": dict(encoder=enc, model_args=deep_args) for rep, enc in representations.items()},
    folds, outcome_space, save_root=config.MODELS_DIR, train_args=deep_train_args, n_jobs=N_JOBS,
)
for name, out in outs.items():
    results_io.save_cv(name, out, vocab_size=tokenizer.get_vocab_size(), **deep_args, **deep_train_args)
    acc = list(out["accuracies"].values())
    print(f"{name}: CV accuracy {np.mean(acc):.4f} ± {np.std(acc):.4f}  (mean ± std over {len(acc)} folds)")
    display(models.epoch_table(out))

encoding:   0%|          | 0/7 [00:00<?, ?it/s]

3 experiments x 5 folds:   0%|          | 0/15 [00:00<?, ?it/s]

deeplowrank__indicator: CV accuracy 0.5255 ± 0.0100  (mean ± std over 5 folds)


,train loss,train accuracy,val loss,val accuracy
epoch,,,,
1,2.387 ± 0.007,0.399 ± 0.003,2.006 ± 0.023,0.485 ± 0.006
2,1.813 ± 0.009,0.518 ± 0.002,1.860 ± 0.018,0.511 ± 0.006
3 *,1.486 ± 0.008,0.581 ± 0.003,1.802 ± 0.035,0.526 ± 0.010
4,1.251 ± 0.013,0.634 ± 0.002,1.854 ± 0.040,0.523 ± 0.007
5,1.078 ± 0.013,0.673 ± 0.003,1.958 ± 0.063,0.510 ± 0.009
6,0.945 ± 0.016,0.708 ± 0.004,2.080 ± 0.046,0.505 ± 0.009
7,0.832 ± 0.014,0.741 ± 0.004,2.234 ± 0.052,0.498 ± 0.008
8,0.736 ± 0.011,0.768 ± 0.003,2.387 ± 0.032,0.481 ± 0.008
9,0.661 ± 0.012,0.791 ± 0.003,2.591 ± 0.090,0.484 ± 0.003


deeplowrank__bow: CV accuracy 0.5256 ± 0.0085  (mean ± std over 5 folds)


,train loss,train accuracy,val loss,val accuracy
epoch,,,,
1,2.391 ± 0.007,0.398 ± 0.002,2.010 ± 0.023,0.483 ± 0.006
2,1.819 ± 0.010,0.516 ± 0.003,1.864 ± 0.017,0.511 ± 0.006
3 *,1.491 ± 0.010,0.579 ± 0.003,1.814 ± 0.034,0.526 ± 0.008
4,1.257 ± 0.012,0.631 ± 0.003,1.858 ± 0.039,0.522 ± 0.008
5,1.081 ± 0.012,0.671 ± 0.003,1.961 ± 0.060,0.509 ± 0.009
6,0.948 ± 0.013,0.707 ± 0.003,2.089 ± 0.040,0.504 ± 0.011
7,0.838 ± 0.014,0.738 ± 0.004,2.250 ± 0.057,0.497 ± 0.009
8,0.742 ± 0.010,0.765 ± 0.003,2.388 ± 0.043,0.481 ± 0.006
9,0.667 ± 0.008,0.788 ± 0.003,2.631 ± 0.085,0.484 ± 0.007


deeplowrank__tfidf: CV accuracy 0.5016 ± 0.0074  (mean ± std over 5 folds)


,train loss,train accuracy,val loss,val accuracy
epoch,,,,
1,2.474 ± 0.004,0.376 ± 0.002,2.185 ± 0.021,0.437 ± 0.006
2,1.983 ± 0.006,0.478 ± 0.002,1.989 ± 0.020,0.486 ± 0.006
3 *,1.705 ± 0.013,0.533 ± 0.003,1.931 ± 0.026,0.498 ± 0.006
4,1.493 ± 0.016,0.573 ± 0.005,1.934 ± 0.038,0.502 ± 0.007
5,1.315 ± 0.017,0.612 ± 0.005,2.007 ± 0.055,0.493 ± 0.009
6,1.169 ± 0.014,0.646 ± 0.003,2.132 ± 0.034,0.492 ± 0.009
7,1.041 ± 0.012,0.678 ± 0.003,2.266 ± 0.046,0.480 ± 0.008
8,0.938 ± 0.014,0.705 ± 0.004,2.455 ± 0.064,0.465 ± 0.008
9,0.845 ± 0.013,0.734 ± 0.004,2.718 ± 0.108,0.466 ± 0.009


### Rank sweep

Same architecture and training, one CV run per rank r (and full rank) for every representation.
All (rank × representation × fold) trainings share one process pool.
Results go to `results/sweeps/rank/` so they stay out of the main comparison in 03_analysis;
reload one later with `results_io.load(name, subdir="sweeps/rank")` and pass it to `models.epoch_table`.

In [38]:
RANKS = [4, 8, 16, 32, 64, 128, None]   # None = full rank (plain deep MLP)
SWEEP_DIR = "sweeps/rank"

sweep = {f"deeplowrank-r{r or 'full'}__{rep}": dict(encoder=enc, model_args={**deep_args, "rank": r})
         for rep, enc in representations.items() for r in RANKS}
sweep_outs = models.run_experiments(
    sweep, folds, outcome_space, save_root=config.MODELS_DIR / SWEEP_DIR,
    train_args=deep_train_args, n_jobs=N_JOBS,
)

rows = []
for name, out in sweep_outs.items():
    cfg = sweep[name]["model_args"]
    results_io.save_cv(name, out, subdir=SWEEP_DIR, vocab_size=tokenizer.get_vocab_size(), **cfg, **deep_train_args)
    rows.append(models.summary_row(out, representation=name.split("__")[1], rank=cfg["rank"] or "full"))
pd.DataFrame(rows).set_index(["representation", "rank"])

encoding:   0%|          | 0/7 [00:00<?, ?it/s]

21 experiments x 5 folds:   0%|          | 0/105 [00:00<?, ?it/s]

params cv acc (best ckpt) best epoch  \
representation rank                                          
indicator      4       55388    0.4905 ± 0.0058  3 5 3 4 4   
               8      102556    0.5100 ± 0.0052  3 4 4 4 4   
               16     196892    0.5166 ± 0.0083  3 3 3 3 3   
               32     385564    0.5255 ± 0.0100  3 3 3 3 3   
               64     762908    0.5309 ± 0.0063  3 3 3 3 3   
               128   1517596    0.5326 ± 0.0077  3 3 3 3 3   
               full  2764828    0.5384 ± 0.0085  2 3 3 3 3   
bow            4       55388    0.4912 ± 0.0074  3 4 5 3 4   
               8      102556    0.5079 ± 0.0058  3 3 3 4 4   
               16     196892    0.5173 ± 0.0064  3 3 3 3 3   
               32     385564    0.5256 ± 0.0085  3 3 3 3 3   
               64     762908    0.5284 ± 0.0090  2 3 3 3 3   
               128   1517596    0.5284 ± 0.0105  3 3 2 3 3   
               full  2764828    0.5394 ± 0.0106  2 3 3 3 3   
tfidf          4       55388    0.4754 ± 0.0076  4 5 5 5 5   
               8      102556    0.4824 ± 0.0038  3 4 3 4 4   
               16     196892    0.4899 ± 0.0107  4 4 4 3 3   
               32     385564    0.5016 ± 0.0074  3 4 4 4 3   
               64     762908    0.5083 ± 0.0078  3 3 3 3 3   
               128   1517596    0.5086 ± 0.0054  3 3 3 3 3   
               full  2764828    0.5192 ± 0.0053  3 3 3 3 3   

                     last-epoch train acc  last-epoch val acc  \
representation rank                                             
indicator      4                   0.6082              0.4708   
               8                   0.7122              0.4752   
               16                  0.7708              0.4753   
               32                  0.8093              0.4767   
               64                  0.8376              0.4783   
               128                 0.8529              0.4780   
               full                0.8963              0.5129   
bow            4                   0.6068              0.4710   
               8                   0.7090              0.4732   
               16                  0.7695              0.4759   
               32                  0.8075              0.4786   
               64                  0.8349              0.4734   
               128                 0.8495              0.4796   
               full                0.8956              0.5118   
tfidf          4                   0.5883              0.4613   
               8                   0.6369              0.4625   
               16                  0.7002              0.4583   
               32                  0.7538              0.4581   
               64                  0.7857              0.4585   
               128                 0.8171              0.4574   
               full                0.8865              0.4740   

                     last-epoch val loss  
representation rank                       
indicator      4                  2.2246  
               8                  2.3685  
               16                 2.6027  
               32                 2.7794  
               64                 2.8387  
               128                2.9087  
               full               2.5732  
bow            4                  2.2281  
               8                  2.3703  
               16                 2.5987  
               32                 2.7804  
               64                 2.8434  
               128                2.9332  
               full               2.5703  
tfidf          4                  2.2400  
               8                  2.3559  
               16                 2.5873  
               32                 2.8839  
               64                 2.9990  
               128                3.1346  
               full               2.8901